#  Advanced RAG Configuration

This notebook shows how to use **external APIs** (OpenAI, Cohere) for advanced RAG features:
- GPT-4 for high-quality responses
- Dense embeddings (not just BM25)
- Cohere reranker for better retrieval

## Prerequisites

1. OpenAI API key: https://platform.openai.com/api-keys
2. Cohere API key (optional): https://dashboard.cohere.ai
3. `.env` file with your keys:
   ```
   OPENAI_API_KEY=sk-...
   COHERE_API_KEY=...
   ```

## Setup: Load Credentials

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env variables
load_dotenv()

# Verify API keys are set
openai_key = os.getenv('OPENAI_API_KEY')
cohere_key = os.getenv('COHERE_API_KEY')

if openai_key:
    print(" OpenAI API key found")
else:
    print(" OPENAI_API_KEY not set in .env")

if cohere_key:
    print(" Cohere API key found")
else:
    print("  COHERE_API_KEY not set (reranker won't work)")

# Setup paths
backend_path = Path("../").resolve()
sys.path.insert(0, str(backend_path))
print(f"\n Working from: {backend_path}")

## Load Premium Configuration

In [ ]:
import yaml
from src.core.factory import RAGPipelineFactory

# Load premium config
config_path = "configs/premium.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

print(" Premium Configuration:")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# Build the premium pipeline
try:
    factory = RAGPipelineFactory.load_config(config_path)
    pipeline = factory.build()
    print(" Premium pipeline built!")
    print(f"\nComponents:")
    print(f"   LLM: {pipeline.llm.__class__.__name__}")
    print(f"   Embedder: {pipeline.embedder.__class__.__name__}")
    print(f"   Retriever: {pipeline.retriever.__class__.__name__}")
    print(f"   Reranker: {pipeline.reranker.__class__.__name__ if pipeline.reranker else 'None'}")
except Exception as e:
    print(f" Error: {e}")

## Example Queries with Premium Setup

In [ ]:
# Prepare documents (same as in quickstart)
sample_docs = {
    "tech_africa.txt": """Africa's tech sector is booming. Nigeria's Lagos is called Africa's Silicon Valley, 
    with companies like Andela, Flutterwave, and Interswitch disrupting fintech and software. 
    Kenya's Nairobi has M-Pesa and numerous startups, while South Africa leads with companies like Takealot.""",
    
    "agriculture.txt": """Agriculture remains central to Africa's economy. Modern agritech startups are 
    revolutionizing farming with precision agriculture, better seeds, and market access. Countries like Rwanda
    are leading agricultural transformation. Organizations like CIMMYT are working to increase crop yields."""
}

docs_dir = Path("data/documents")
docs_dir.mkdir(parents=True, exist_ok=True)

for name, content in sample_docs.items():
    (docs_dir / name).write_text(content)
    print(f"Created {name}")

# Ingest
print("\n Ingesting documents...")
for doc_path in ["data/documents/tech_africa.txt", "data/documents/agriculture.txt"]:
    try:
        result = pipeline.ingest(doc_path)
        print(f" {Path(doc_path).name}: {result['chunks_count']} chunks")
    except Exception as e:
        print(f"Error: {e}")

print(f"\nTotal documents indexed: {pipeline.vector_store.count()}")

In [ ]:
# High-quality query with OpenAI
question = "What are the main tech innovations driving Africa's economy forward?"

print(f" {question}\n")
print(" Using premium setup (GPT-4 + Dense embeddings + Cohere reranker)\n")

try:
    response = pipeline.query(
        question,
        top_k=5,
        temperature=0.7  # Balanced creativity
    )
    
    print(" High-Quality Answer:")
    print("=" * 80)
    print(response['answer'])
    print("=" * 80)
    
    print(f"\n Sources retrieved and reranked:")
    for i, source in enumerate(response['sources'], 1):
        print(f"\n[Source {i}]")
        print(f"{source['content'][:200]}...")
        
except Exception as e:
    print(f" Error: {e}")

## Create Custom Configuration

In [ ]:
# Create a custom config (hybrid approach)
custom_config = """
ingestion:
  loader: "text_loader"
  chunker: "basic_chunker"
  chunk_size: 512
  overlap: 50

indexation:
  embedder: "dense"        # Use SBERT for dense embeddings
  vector_store: "chroma"   # Chroma for better performance
  batch_size: 32

retrieval:
  strategy: "hybrid"       # Combine dense + BM25
  top_k: 5
  reranker: "cohere"       # Rerank with Cohere

generation:
  llm: "openai"
  model: "gpt-4o-mini"  # Fast and capable
  temperature: 0.7
  max_tokens: 1000
  prompt_template: "citations"  # Include source citations
"""

# Save custom config
custom_config_path = "configs/custom_hybrid.yaml"
Path(custom_config_path).write_text(custom_config)

print(f" Custom config saved to {custom_config_path}")
print(f"\nConfig content:\n{custom_config}")

In [ ]:
# Load and test custom config
try:
    custom_factory = RAGPipelineFactory.load_config(custom_config_path)
    custom_pipeline = custom_factory.build()
    print(" Custom pipeline built!")
    
    # Use existing documents
    question = "How is agriculture evolving in Africa?"
    response = custom_pipeline.query(question, top_k=3)
    
    print(f"\nCustom Pipeline - {question}")
    print(f"Answer: {response['answer'][:300]}...")
except Exception as e:
    print(f"Note: {e}")
    print("(Custom config requires documents to be ingested again)")

## Test with API Directly

In [ ]:
import requests
import json
from time import sleep

# Make sure the API is running
# In another terminal: python -m uvicorn src.api.main:app --reload

API_URL = "http://localhost:8000"

try:
    # Health check
    response = requests.get(f"{API_URL}/health", timeout=2)
    print(" API is running!")
except:
    print(" API not reachable at", API_URL)
    print("Start it with: python -m uvicorn src.api.main:app --reload")

In [ ]:
# Test /ingest endpoint
ingest_payload = {
    "source": "data/documents/tech_africa.txt",
    "loader_name": "text_loader",
    "chunker_name": "basic_chunker"
}

try:
    response = requests.post(
        f"{API_URL}/ingest",
        json=ingest_payload,
        timeout=10
    )
    if response.status_code == 200:
        result = response.json()
        print(f" Ingest successful")
        print(json.dumps(result, indent=2))
    else:
        print(f" Error {response.status_code}: {response.text}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Test /query endpoint
query_payload = {
    "question": "What tech companies are disrupting Africa?",
    "top_k": 3,
    "streaming": False
}

try:
    response = requests.post(
        f"{API_URL}/query",
        json=query_payload,
        timeout=15
    )
    if response.status_code == 200:
        result = response.json()
        print(f" Query successful\n")
        print(f"Answer: {result.get('answer', 'N/A')}")
        print(f"\nSources: {len(result.get('sources', []))} documents")
    else:
        print(f" Error {response.status_code}: {response.text}")
except Exception as e:
    print(f"Error: {e}")

## Performance Comparison

In [ ]:
import time

# Test query performance
test_questions = [
    "What are Africa's tech innovations?",
    "How is agriculture changing?",
    "Which companies are leading?"
]

print("⏱  Performance Test\n")
print(f"{'Question':<40} {'Time (s)':<10} {'Sources'}")
print("-" * 60)

for question in test_questions:
    start = time.time()
    try:
        response = pipeline.query(question, top_k=3)
        elapsed = time.time() - start
        sources = len(response.get('sources', []))
        print(f"{question:<40} {elapsed:<10.2f} {sources}")
    except Exception as e:
        print(f"{question:<40} ERROR: {str(e)[:20]}")

## Summary

You've learned:
- ✅ Using premium configurations with external APIs
- ✅ Dense embeddings + BM25 hybrid retrieval
- ✅ Reranking with Cohere
- ✅ Creating custom configurations
- ✅ Testing the API endpoints
- ✅ Performance benchmarking

### Next steps:
1. Deploy with `python -m uvicorn src.api.main:app`
2. Connect to your frontend
3. Monitor performance in production
4. Fine-tune parameters based on your use cases